# Lecture 7.2: Your First Subagent

The Bike Repair Co-op is a cooperative workshop run by volunteers. Members drop off bikes, volunteers log a work order for each repair, and the shop keeps a small history log on every job. Forty work orders are waiting in the shop folder, and five of them describe bikes that are both **urgent** and a **safety issue**. Those five are the ones that must be fixed before anything leaves the workshop.

In this notebook you give an agent the job of finding those five orders, and you do it twice:

1. **Main agent alone.** The agent reads every work order itself.
2. **With a subagent.** The main agent delegates the reading to a subagent and only receives the subagent's final report.

You will watch both runs message by message, then compare how much work each one put on the main agent.

### The mental model to keep in mind

`query()` is not a function that returns a value. It returns an **async stream of messages** while the agent loop runs on its own. Tool calls and tool results arrive as separate messages in that stream, and the final answer arrives last on the `ResultMessage`. A `ToolResultBlock` arrives inside a `UserMessage`, not inside an `AssistantMessage`. Delegation adds one more idea on top: some messages in the stream now come from *inside* a subagent, and a field called `parent_tool_use_id` tells you which.

### What the cells do

| Cells | Purpose |
|---|---|
| 1 to 3 | Setup: install the SDK, load your API key, choose the model |
| 4 to 5 | Create the Bike Repair Co-op files and compute the correct answer from them |
| 6 | A helper that runs an agent and logs every message in the stream |
| 7 | Run the main agent alone |
| 8 to 9 | Run with a subagent, then inspect the call and what came back |
| 10 to 11 | Compare the two runs and count where the work happened |

Agents are not deterministic. The wording of each answer, the number of messages and the token counts will differ from run to run, and that is normal. In this notebook you look for patterns, not exact numbers.

## Cell 1: Install the Claude Agent SDK

This cell installs the `claude-agent-sdk` Python package. It is the only package this notebook needs.

| Part | Meaning |
|---|---|
| `!pip install` | The leading `!` tells Colab to run the line in the shell instead of as Python |
| `claude-agent-sdk==0.2.174` | The package, pinned to one exact version |
| `-q` | Quiet mode, so the cell prints only errors and not a long install log |

**Why pin the version?** Pinning keeps every example in this notebook reproducible. The SDK ships with the Claude Code command line tool bundled inside it, so a new release can change agent behavior. If you want the latest release, run `pip install -U claude-agent-sdk` instead, or substitute any version you prefer. Just expect newer versions to behave differently from the videos.

If the package is already installed in your current Colab session, pip will see it and finish almost immediately. Run the cell and wait for it to finish before moving on.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.174 -q

## Cell 2: Load your API key from Colab Secrets

The SDK reads your Anthropic API key from the `ANTHROPIC_API_KEY` environment variable. This cell copies the key from Colab's secret store into that variable. The key never appears in the notebook itself, so you can share the notebook safely.

**Add the secret in Colab (one time):**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to exactly `ANTHROPIC_API_KEY`.
4. Paste your API key as the value.
5. Switch **Notebook access** on for this notebook.

You create the key in the Claude Console at [platform.claude.com](https://platform.claude.com/) under **API Keys**.

**Running locally instead of Colab?** Skip this cell and set the environment variable in your terminal before you start Jupyter, for example `export ANTHROPIC_API_KEY=your-key-here`.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

This cell stores the model name in one variable, `MODEL_NAME`. Every agent call later in the notebook passes `model=MODEL_NAME`, so **changing this one line changes the model for the entire notebook**.

This notebook uses Claude Haiku 5.5 because it is fast and lightweight, which suits a demo that runs several agents back to back. Model names change with new releases, so the comment in the cell links to the models page where you can find the current list.

Subagents inherit the main agent's model unless you say otherwise, so this one variable also decides which model the subagent uses.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create the Bike Repair Co-op files

This cell builds the workshop folder that both agent runs will work on. Everything is created in code, so every learner starts with the same files.

```
/content/bikecoop/
    work_orders/WO-101.txt ... WO-140.txt    (40 work orders)
    inventory.csv                            (parts stock levels)
    volunteers.txt                           (volunteer skills and shifts)
```

**The work orders** are the files that matter in this notebook. Each one is a short text file of roughly 1,200 to 1,500 characters. The top holds the structured fields (bike, owner, problem, priority, safety issue, status, assigned volunteer) and the rest is a repair history log, so each file has real bulk, just like a real job file.

**Design decisions:**

- **A fixed random seed.** `random.Random(7)` makes the generator produce identical files on every run, so the correct answer never changes.
- **Exactly five orders are `urgent` with a safety issue of `yes`.** These five are the ground-truth answer.
- **Deliberate look-alikes.** Five orders are `urgent` but have no safety issue, and six have a safety issue but only `high` priority. An agent that skims the files will confuse them with the real five, so a correct answer proves the agent read carefully.

The `inventory.csv` and `volunteers.txt` files are created now but are not used in this notebook.

In [ ]:
import os
import random
import shutil

BASE = "/content/bikecoop"
WO_DIR = f"{BASE}/work_orders"

# Start from a clean folder so every run produces identical files.
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(WO_DIR)

rng = random.Random(7)

bikes = [
    "Blue hybrid city bike", "Red steel road bike", "Green mountain bike",
    "Black folding bike", "Yellow cargo bike", "Silver commuter bike",
    "White step-through bike", "Orange kids bike, 20 inch wheels",
    "Grey gravel bike", "Purple tandem", "Teal touring bike",
    "Cream vintage three-speed", "Black BMX bike", "Maroon e-bike",
    "Lime green trail bike", "Pink kids bike, 16 inch wheels",
    "Brown utility bike", "Navy racing bike", "Olive hardtail mountain bike",
    "Gold single-speed bike",
]

owners = [
    "Anita Rao", "Marcus Bell", "Priya Nair", "Tom Okafor", "Lena Fischer",
    "Dev Malhotra", "Sofia Alvarez", "Kwame Mensah", "Yuki Tanaka",
    "Ravi Iyer", "Hannah Cole", "Omar Haddad", "Grace Liu", "Nikhil Shah",
    "Elena Petrova", "Samuel Wright", "Meera Pillai", "Jonas Berg",
    "Fatima Khan", "Arjun Menon",
]

volunteers = [
    ("Asha", "wheels and brakes", "Monday and Thursday"),
    ("Ben", "gears and drivetrain", "Tuesday and Saturday"),
    ("Chitra", "frames and welding", "Wednesday and Saturday"),
    ("Dinesh", "e-bike electrics", "Friday"),
    ("Esha", "general servicing", "Monday and Wednesday"),
    ("Farid", "kids bikes and fitting", "Thursday and Sunday"),
]

urgent_safety_problems = [
    "Front brake lever pulls to the handlebar with almost no stopping power",
    "Cracked weld near the fork crown",
    "Rear brake cable frayed and close to snapping",
    "Chain keeps jumping under load and has locked up once",
    "Front wheel wobbles because a bearing cone has come loose",
    "Handlebar stem clamp slips while riding",
    "Crank arm is loose on the spindle",
    "Rear wheel has a broken spoke and is badly out of true",
]

urgent_no_safety_problems = [
    "Owner wants a full tune-up before a charity ride on Saturday",
    "Paint scratched and rust starting at the dropouts",
    "Bell missing and basket bracket bent",
    "Saddle torn and owner starts a new commute tomorrow",
    "Kickstand broken so the bike keeps falling over",
]

high_safety_problems = [
    "Brake pads worn down to the wear line",
    "Rear light will not hold a charge",
    "Tyre sidewall bulging slightly",
    "Gear cable sticky and skipping gears",
    "Pedal bearing feels gritty",
    "Rear reflector missing",
]

routine_problems = [
    "Squeaky bottom bracket", "Flat tyre needs a new tube",
    "Gear indicator out of alignment", "Mudguard rattles over bumps",
    "Grips worn smooth", "Chain needs cleaning and lubrication",
    "Saddle height will not stay put", "Water bottle cage is loose",
    "Seasonal service requested", "Small wobble, wheel needs truing",
    "Brakes squeal in wet weather", "Handlebar tape unwinding",
]

history_notes = [
    "Intake check done and frame inspected for visible damage.",
    "Owner phoned to confirm the best day for collection.",
    "Spare parts shelf checked for matching sizes.",
    "Workshop stand booked for the next free slot.",
    "Photo of the fault added to the job folder.",
    "Test ride completed in the yard, no further faults noted.",
    "Cleaned the drivetrain before inspection started.",
    "Tyre pressure and quick-release levers checked.",
    "Owner asked for a quote before any parts are ordered.",
    "Handover checklist printed and clipped to the job card.",
    "Bike tagged and moved to the repair bay.",
    "Noted that the owner collects on weekday evenings.",
]

# Decide which order gets which role, using the seeded generator.
ids = list(range(101, 141))
rng.shuffle(ids)
urgent_safety_ids = ids[:5]
urgent_no_ids = ids[5:10]
high_safety_ids = ids[10:16]
routine_ids = ids[16:]

problem_for = {}
priority_for = {}
safety_for = {}

for number, problem in zip(urgent_safety_ids, rng.sample(urgent_safety_problems, 5)):
    problem_for[number], priority_for[number], safety_for[number] = problem, "urgent", "yes"

for number, problem in zip(urgent_no_ids, urgent_no_safety_problems):
    problem_for[number], priority_for[number], safety_for[number] = problem, "urgent", "no"

for number, problem in zip(high_safety_ids, high_safety_problems):
    problem_for[number], priority_for[number], safety_for[number] = problem, "high", "yes"

for number in routine_ids:
    problem_for[number] = rng.choice(routine_problems)
    priority_for[number] = rng.choice(["high", "normal", "normal", "low"])
    safety_for[number] = "no"

for number in range(101, 141):
    work_order_id = f"WO-{number}"
    volunteer_name = rng.choice(volunteers)[0]
    status = rng.choice(["open", "in progress", "waiting for parts"])

    text = (
        f"Work order: {work_order_id}\n"
        f"Bike: {rng.choice(bikes)}\n"
        f"Owner: {rng.choice(owners)}\n"
        f"Problem: {problem_for[number]}\n"
        f"Priority: {priority_for[number]}\n"
        f"Safety issue: {safety_for[number]}\n"
        f"Status: {status}\n"
        f"Assigned volunteer: {volunteer_name}\n"
        f"\n"
        f"Repair history log:\n"
    )

    # Add history lines until the file reaches its target size.
    target_length = rng.randint(1250, 1450)
    while True:
        entry = (
            f"2026-{rng.randint(1, 9):02d}-{rng.randint(1, 28):02d} | "
            f"{rng.choice(volunteers)[0]} | {rng.choice(history_notes)}\n"
        )
        if len(text) + len(entry) > target_length:
            break
        text += entry

    with open(f"{WO_DIR}/{work_order_id}.txt", "w", encoding="utf-8") as file:
        file.write(text)

# Supporting files, created now for later use.
parts = [
    ("brake pads", 14, 10), ("brake cables", 6, 8), ("inner tubes 700c", 22, 12),
    ("inner tubes 26 inch", 9, 12), ("chains", 5, 6), ("chain rings", 3, 4),
    ("spokes assorted", 120, 60), ("bar tape", 11, 6), ("saddles", 4, 3),
    ("pedals pairs", 7, 5), ("headset bearings", 8, 6), ("gear cables", 10, 8),
    ("rear lights", 12, 8), ("kickstands", 2, 4), ("grips pairs", 9, 6),
]

with open(f"{BASE}/inventory.csv", "w", encoding="utf-8") as file:
    file.write("part,stock,reorder_level\n")
    for part, stock, reorder_level in parts:
        file.write(f"{part},{stock},{reorder_level}\n")

with open(f"{BASE}/volunteers.txt", "w", encoding="utf-8") as file:
    for name, skills, shifts in volunteers:
        file.write(f"{name} | skills: {skills} | shifts: {shifts}\n")

print("Bike Repair Co-op files created.")

Bike Repair Co-op files created.


## Cell 5: Verify the files and compute the ground truth

Before any agent runs, this cell proves the files exist and works out the correct answer **using plain Python**, with no AI involved.

It does five things:

1. Prints the folder tree. The 40 work orders are summarized on one line instead of listed one by one.
2. Prints the smallest and largest file sizes, so you can confirm each file has real bulk.
3. Prints one complete sample work order, so you can see the fields and the history log the agents will read.
4. Prints `inventory.csv` and `volunteers.txt` in full. They are short, and you will not use them in this notebook, but you can see what the rest of the shop data looks like.
5. Reads the `Priority` and `Safety issue` fields from every file and collects the orders that are `urgent` with a safety issue of `yes`.

The result is stored in `GROUND_TRUTH`, a dictionary of work order ID to problem. Later cells compare each agent's answer against it. This is **filesystem ground truth**: instead of judging an agent's answer by how convincing it sounds, you check it against facts computed directly from the files.

The cell ends with an `assert` that exactly five orders match. If it fails, rerun Cell 4.

In [ ]:
import glob
import re

# 1. Folder tree
print("bikecoop/")
for name in sorted(os.listdir(BASE)):
    path = f"{BASE}/{name}"
    if os.path.isdir(path):
        files = sorted(os.listdir(path))
        print(f"    {name}/  ({len(files)} files, {files[0]} ... {files[-1]})")
    else:
        print(f"    {name}")

# 2. File sizes
sizes = [os.path.getsize(path) for path in glob.glob(f"{WO_DIR}/WO-*.txt")]
print(f"\nWork order sizes: smallest {min(sizes)} bytes, largest {max(sizes)} bytes")

# 3. One sample work order
print("\n--- Sample work order: WO-101 ---")
with open(f"{WO_DIR}/WO-101.txt", encoding="utf-8") as file:
    print(file.read())

# 4. The supporting files (short, so we print them in full)
for name in ("inventory.csv", "volunteers.txt"):
    print(f"\n--- {name} ---")
    with open(f"{BASE}/{name}", encoding="utf-8") as file:
        print(file.read())


# 5. Ground truth, computed from the files with plain Python
def read_field(text, label):
    match = re.search(rf"^{label}:\s*(.+)$", text, re.MULTILINE)
    return match.group(1).strip() if match else None


GROUND_TRUTH = {}
for path in sorted(glob.glob(f"{WO_DIR}/WO-*.txt")):
    with open(path, encoding="utf-8") as file:
        text = file.read()
    if read_field(text, "Priority") == "urgent" and read_field(text, "Safety issue") == "yes":
        GROUND_TRUTH[read_field(text, "Work order")] = read_field(text, "Problem")

print("--- Ground truth: urgent AND safety issue ---")
for work_order_id, problem in GROUND_TRUTH.items():
    print(f"{work_order_id}: {problem}")

assert len(GROUND_TRUTH) == 5, "Expected exactly five urgent safety orders. Rerun Cell 4."

bikecoop/
    inventory.csv
    volunteers.txt
    work_orders/  (40 files, WO-101.txt ... WO-140.txt)

Work order sizes: smallest 1220 bytes, largest 1408 bytes

--- Sample work order: WO-101 ---
Work order: WO-101
Bike: Green mountain bike
Owner: Marcus Bell
Problem: Rear light will not hold a charge
Priority: high
Safety issue: yes
Status: in progress
Assigned volunteer: Chitra

Repair history log:
2026-05-21 | Esha | Bike tagged and moved to the repair bay.
2026-08-10 | Farid | Cleaned the drivetrain before inspection started.
2026-06-01 | Dinesh | Test ride completed in the yard, no further faults noted.
2026-03-20 | Asha | Tyre pressure and quick-release levers checked.
2026-01-07 | Chitra | Spare parts shelf checked for matching sizes.
2026-04-13 | Dinesh | Tyre pressure and quick-release levers checked.
2026-02-06 | Dinesh | Cleaned the drivetrain before inspection started.
2026-09-09 | Ben | Cleaned the drivetrain before inspection started.
2026-09-09 | Farid | Cleaned the dri

## Cell 6: A helper that runs an agent and logs every message

You will run the agent twice, so this cell defines one helper, `run_and_log(prompt, tools)`, and reuses it. The helper has three jobs.

1. **Run the agent.** It calls `query()` with `allowed_tools=tools`, `model=MODEL_NAME` and `cwd="/content"`. The `cwd` option sets the agent's working directory to `/content`, where the Bike Repair Co-op files live.
2. **Keep every message** in a list, so later cells can inspect the stream after the run is over.
3. **Print one compact line per message.** Each line shows the message type, its `parent_tool_use_id` (shortened to the last six characters) and the types of content blocks inside it.

**How to read the log lines:**

| Message type | What it means |
|---|---|
| `AssistantMessage` | Something the model produced: text, or a `ToolUseBlock` that requests a tool call |
| `UserMessage` | Tool results being fed back to the model. The `ToolResultBlock` lives here |
| `ResultMessage` | The final message of the run: the answer, token usage and cost |
| Other types | Housekeeping events such as session setup. Log lines show their type name and subtype, and you can ignore them for now |

**`parent_tool_use_id` is the key field for this lecture.** For messages from the main agent it is `None`. For messages produced *inside* a subagent it holds the ID of the tool call that started that subagent. That is how one flat stream of messages can be sorted into who did what.

The helper returns the list of messages and the `ResultMessage`. If the run ends with an error, the SDK raises after yielding the error result, so the helper catches the exception and still returns what it collected.

In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    ToolUseBlock,
    ToolResultBlock,
)


def short_id(value):
    """Shorten a tool call ID so log lines stay compact."""
    if value is None:
        return "None"
    return "..." + value[-6:]


def block_types(message):
    """List the content block types inside a message."""
    content = message.content
    if isinstance(content, list):
        return [type(block).__name__ for block in content]
    if isinstance(content, str):
        return ["str"]
    return []


async def run_and_log(prompt, tools):
    """Run one agent, keep every message, and print one line per message."""
    messages = []
    result = None
    try:
        async for message in query(
            prompt=prompt,
            options=ClaudeAgentOptions(
                allowed_tools=tools,
                model=MODEL_NAME,
                cwd="/content",
            ),
        ):
            messages.append(message)
            line = f"{len(messages):>3}  {type(message).__name__:<22}"

            if isinstance(message, (AssistantMessage, UserMessage)):
                line += f" parent={short_id(message.parent_tool_use_id):<11}"
                line += f" blocks={block_types(message)}"
            elif isinstance(message, ResultMessage):
                line += f" subtype={message.subtype}"
                result = message
            else:
                line += f" subtype={getattr(message, 'subtype', '')}"

            print(line)
    except Exception as error:
        print(f"Run ended with an error: {error}")
    return messages, result

## Cell 7: Run the main agent alone

This is the baseline. The goal prompt is the same one you will reuse in the next run.

```
Read every work order in /content/bikecoop/work_orders and list the five most urgent ones,
meaning urgent priority with a safety issue. Give each one's ID and problem.
```

The prompt states a **goal**, not a procedure. It does not tell the agent which tool to call or in what order. The agent decides that on its own.

**Tools: `["Read", "Glob", "Grep"]`.** These are read-only tools, so the agent can look at files but cannot change anything. `Glob` finds files by pattern, `Grep` searches inside files, and `Read` opens a file. There is no `Agent` tool in this list, so the main agent has to do all the reading itself.

After the run, the cell does three things:

1. Prints the final answer from `ResultMessage.result`.
2. Checks which of the five ground-truth IDs appear in that answer.
3. Prints the **main agent's usage**, the token counts reported on `ResultMessage.usage`.

**What to look for:** many `ToolUseBlock` and `ToolResultBlock` lines in the log, one for each file the agent opened. Every one of those results landed in the main agent's own context. Then check the ground-truth count. Your exact output will differ from anyone else's, and that is normal for a real agent.

In [ ]:
GOAL = (
    "Read every work order in /content/bikecoop/work_orders and list the five "
    "most urgent ones, meaning urgent priority with a safety issue. "
    "Give each one's ID and problem."
)

solo_messages, solo_result = await run_and_log(GOAL, ["Read", "Glob", "Grep"])

print("\n--- Final answer (main agent alone) ---")
solo_answer = (solo_result.result if solo_result else None) or ""
print(solo_answer or "(no answer returned)")

found = [work_order_id for work_order_id in GROUND_TRUTH if work_order_id in solo_answer]
print(f"\nGround-truth IDs found: {len(found)} of {len(GROUND_TRUTH)}")
missing = [work_order_id for work_order_id in GROUND_TRUTH if work_order_id not in found]
if missing:
    print(f"Missing: {missing}")

print("\n--- Main agent usage ---")
if solo_result and solo_result.usage:
    for key in (
        "input_tokens",
        "cache_read_input_tokens",
        "cache_creation_input_tokens",
        "output_tokens",
    ):
        print(f"{key:<30}{solo_result.usage.get(key, 0)}")

  1  SystemMessage          subtype=init
  2  AssistantMessage       parent=None        blocks=['ThinkingBlock']
  3  AssistantMessage       parent=None        blocks=['ToolUseBlock']
  4  UserMessage            parent=None        blocks=['ToolResultBlock']
  5  SystemMessage          subtype=thinking_tokens
  6  SystemMessage          subtype=thinking_tokens
  7  AssistantMessage       parent=None        blocks=['ThinkingBlock']
  8  AssistantMessage       parent=None        blocks=['ToolUseBlock']
  9  SystemMessage          subtype=permission_denied
 10  UserMessage            parent=None        blocks=['ToolResultBlock']
 11  AssistantMessage       parent=None        blocks=['ToolUseBlock']
 12  UserMessage            parent=None        blocks=['ToolResultBlock']
 13  SystemMessage          subtype=thinking_tokens
 14  SystemMessage          subtype=thinking_tokens
 15  AssistantMessage       parent=None        blocks=['ThinkingBlock']
 16  AssistantMessage       parent=None       

## Cell 8: Run again, this time with a subagent

Now the same goal runs a second time, with two changes.

**Change 1: add `"Agent"` to `allowed_tools`.** That one word is all it takes to turn delegation on. The main agent can now start the built-in `general-purpose` subagent, and you do not need to define any agents yourself.

**Change 2: the prompt asks for a subagent.** The delegated prompt is the same goal, preceded by:

```
Use a subagent to read the work orders, then give me the final answer.
```

Naming the subagent in the prompt is called **explicit invocation**. Without it, a small model such as Haiku may decide to just read the files itself and skip delegation, which would make this demo pointless. Explicit invocation is a pattern you will build on as you work with subagents.

**Finding the delegation.** In the stream, the delegation is an ordinary `ToolUseBlock` inside an `AssistantMessage`, and the tool is named `"Agent"`. The code also matches `"Task"`, because the SDK documentation notes that this tool appears under the name `"Task"` in some places, such as the session's tool list and permission denials. Matching both names keeps the code working across versions.

**What the cell prints about the call:**

- The call's `id`. Every message from inside the subagent will carry this value as its `parent_tool_use_id`.
- The `subagent_type`.
- The **complete `input` dictionary**, key by key. This cell prints every key that is really present instead of assuming any names, so you see exactly what the main agent sent. The key that matters most is the prompt the main agent wrote for the subagent. It is the **only thing the subagent receives** from the main agent. The subagent does not see the main agent's conversation.

**What to look for:** an `Agent` tool call in the log, followed by a block of messages whose parent ID matches that call, and then a single result coming back to the main agent. If the cell reports that no delegation happened, run it again.

In [ ]:
DELEGATED_PROMPT = (
    "Use a subagent to read the work orders, then give me the final answer. " + GOAL
)

deleg_messages, deleg_result = await run_and_log(
    DELEGATED_PROMPT, ["Read", "Glob", "Grep", "Agent"]
)


def find_agent_call(messages):
    """Return the first Agent ToolUseBlock sent by the main agent."""
    for message in messages:
        if isinstance(message, AssistantMessage) and message.parent_tool_use_id is None:
            for block in message.content:
                if isinstance(block, ToolUseBlock) and block.name in ("Agent", "Task"):
                    return block
    return None


AGENT_CALL = find_agent_call(deleg_messages)

print("\n--- The delegation ---")
if AGENT_CALL is None:
    print("No Agent call found. The main agent answered without delegating. Run this cell again.")
else:
    print(f"Agent call id : {AGENT_CALL.id}")
    print(f"Tool name     : {AGENT_CALL.name}")
    print(f"Input keys    : {list(AGENT_CALL.input.keys())}")
    print(f"subagent_type : {AGENT_CALL.input.get('subagent_type', '(not present)')}")

    print("\nPrompt the main agent wrote for the subagent:")
    print(AGENT_CALL.input.get("prompt", "(no 'prompt' key, check the input keys above)"))

    print("\nOther input values:")
    for key, value in AGENT_CALL.input.items():
        if key not in ("subagent_type", "prompt"):
            print(f"  {key}: {str(value)[:200]}")

  1  SystemMessage          subtype=init
  2  SystemMessage          subtype=thinking_tokens
  3  SystemMessage          subtype=thinking_tokens
  4  AssistantMessage       parent=None        blocks=['ThinkingBlock']
  5  AssistantMessage       parent=None        blocks=['ToolUseBlock']
  6  SystemMessage          subtype=background_tasks_changed
  7  TaskStartedMessage     subtype=task_started
  8  UserMessage            parent=None        blocks=['ToolResultBlock']
  9  AssistantMessage       parent=None        blocks=['TextBlock']
 10  AssistantMessage       parent=...Xem5Lc   blocks=['ThinkingBlock']
 11  AssistantMessage       parent=...Xem5Lc   blocks=['ToolUseBlock']
 12  TaskProgressMessage    subtype=task_progress
 13  SystemMessage          subtype=permission_denied
 14  UserMessage            parent=...Xem5Lc   blocks=['ToolResultBlock']
 15  AssistantMessage       parent=...Xem5Lc   blocks=['ToolUseBlock']
 16  TaskProgressMessage    subtype=task_progress
 17  UserMessage  

## Cell 9: The result of the Agent call

The subagent did its reading out of sight. This cell shows the tool result the main agent received for the `Agent` call, and then the main agent's final answer.

The result of the `Agent` call arrives like the result of any other tool: as a `ToolResultBlock` inside a `UserMessage`. Two details identify it:

- Its `tool_use_id` equals the `id` of the `Agent` call from Cell 8.
- It sits in a message whose `parent_tool_use_id` is `None`, because it was delivered to the main agent, not produced inside the subagent.

**Handling the content.** The `content` of a `ToolResultBlock` can be either a plain string or a list of blocks, so the helper function `result_text()` handles both. You may also see a line such as `agentId: ...` at the end of the text. That is an identifier for the subagent that ran. You can ignore it for now.

The cell prints the text, truncated if it is long, and checks which ground-truth IDs appear in it. Then it prints the main agent's final answer and checks that too.

**Heads up: subagents run in the background by default.** When that happens, the tool result for the `Agent` call is only a short note saying the subagent was started. The report does not come back in that tool result. The final answer printed below is what the main agent produced.

Either way, the key point stands: the file reads and searches happened **inside the subagent**, and none of them entered the main agent's own context.

**What to look for:** the text of the tool result, which may be only a launch note, and then the main agent's final answer with its ground-truth count.

In [ ]:
def result_text(block):
    """Return the text of a ToolResultBlock, whether content is a string or a list."""
    content = block.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            part.get("text", "") for part in content if isinstance(part, dict)
        )
    return ""


AGENT_RESULT = None
if AGENT_CALL is not None:
    for message in deleg_messages:
        if isinstance(message, UserMessage) and isinstance(message.content, list):
            for block in message.content:
                if isinstance(block, ToolResultBlock) and block.tool_use_id == AGENT_CALL.id:
                    AGENT_RESULT = (message, block)

print("--- The subagent's result, as the main agent received it ---")
if AGENT_RESULT is None:
    print("No tool result matched the Agent call id.")
else:
    result_message, result_block = AGENT_RESULT
    text = result_text(result_block)
    print(f"parent_tool_use_id of this message: {result_message.parent_tool_use_id}")
    print(f"is_error: {result_block.is_error}")
    print(f"Length: {len(text)} characters\n")
    print(text[:2000])
    if len(text) > 2000:
        print(f"\n... (truncated, {len(text)} characters in total)")

    found = [work_order_id for work_order_id in GROUND_TRUTH if work_order_id in text]
    print(f"\nGround-truth IDs in the subagent's report: {len(found)} of {len(GROUND_TRUTH)}")

print("\n--- Final answer from the main agent ---")
deleg_answer = (deleg_result.result if deleg_result else None) or ""
print(deleg_answer or "(no answer returned)")

found_final = [work_order_id for work_order_id in GROUND_TRUTH if work_order_id in deleg_answer]
print(f"\nGround-truth IDs in the final answer: {len(found_final)} of {len(GROUND_TRUTH)}")

--- The subagent's result, as the main agent received it ---
parent_tool_use_id of this message: None
is_error: None
Length: 1031 characters

Async agent launched successfully. (This tool result is internal metadata — never quote or paste any part of it, including the agentId below, into a user-facing reply.)
agentId: af3a443bb29d5cfb6 (internal ID - do not mention to user. Use SendMessage with to: 'af3a443bb29d5cfb6', summary: '<5-10 word recap>' to continue this agent.)
The agent is working in the background. You will be notified automatically when it completes. You know nothing about its results until that notification arrives — do not report, assume, or predict them; continue other work or respond to the user in the meantime.
Do not duplicate this agent's work — avoid working with the same files or topics it is using.
output_file: /tmp/claude-0/-content/009f1820-d575-4888-8510-c5ff21524307/tasks/af3a443bb29d5cfb6.output
Do NOT Read or tail this file via the shell tool — it is the f

## Cell 10: Compare the two runs

This cell puts the two runs side by side in one small table. It uses four measurements from each `ResultMessage`.

| Row | Where it comes from | What it covers |
|---|---|---|
| Main agent input-side tokens | `usage`: `input_tokens` + `cache_read_input_tokens` + `cache_creation_input_tokens` | The main agent only |
| Main agent output tokens | `usage["output_tokens"]` | The main agent only |
| `total_cost_usd` | `ResultMessage.total_cost_usd` | The whole run, **including the subagent** |
| Models in the breakdown | `ResultMessage.model_usage` | The whole run, **including the subagent** |

**Why add three numbers for input-side tokens?** With prompt caching, the text the model reads on each step can be counted as new input, as a cache write or as a cache read. Adding the three gives the total amount of text the main agent's model had to read.

**The most important thing to understand in this notebook:** `usage` counts only the **main agent's own loop**. It does not include tokens spent inside the subagent. Cost and the per-model breakdown, in contrast, include the subagent. So when delegation works well, you expect the main agent's own input-side tokens to be smaller, because 40 file reads never entered its context. The total cost can still be similar or even higher, because the subagent did the reading and was billed for it.

`total_cost_usd` is a **client-side estimate** computed by the SDK from a price table. Use it to compare runs during development, not as billing data.

**What to look for:** whether the main agent's input-side tokens are lower in the delegated run, and whether `total_cost_usd` tells a different story from `usage`. Token counts vary from run to run, so look at the direction of the difference and not at the exact values.

In [ ]:
def input_side(usage):
    """Everything the main agent's model read: new input plus cache reads and writes."""
    usage = usage or {}
    return sum(
        usage.get(key, 0) or 0
        for key in (
            "input_tokens",
            "cache_read_input_tokens",
            "cache_creation_input_tokens",
        )
    )


def summarize(result):
    if result is None:
        return None, None, None, 0, []
    models = list((result.model_usage or {}).keys())
    return (
        input_side(result.usage),
        (result.usage or {}).get("output_tokens", 0),
        result.total_cost_usd,
        len(models),
        models,
    )


solo_in, solo_out, solo_cost, solo_n, solo_models = summarize(solo_result)
deleg_in, deleg_out, deleg_cost, deleg_n, deleg_models = summarize(deleg_result)


def fmt(value, decimals=0):
    if value is None:
        return "n/a"
    return f"{value:,.{decimals}f}"


print(f"{'Measurement':<44}{'Alone':>14}{'With subagent':>16}")
print("-" * 74)
print(f"{'Main agent input-side tokens':<44}{fmt(solo_in):>14}{fmt(deleg_in):>16}")
print(f"{'Main agent output tokens':<44}{fmt(solo_out):>14}{fmt(deleg_out):>16}")
print(f"{'total_cost_usd (estimate, whole run)':<44}{fmt(solo_cost, 4):>14}{fmt(deleg_cost, 4):>16}")
print(f"{'Models in model_usage (whole run)':<44}{solo_n:>14}{deleg_n:>16}")

print("\nmodel_usage keys, main agent alone :", solo_models)
print("model_usage keys, with a subagent  :", deleg_models)

Measurement                                          Alone   With subagent
--------------------------------------------------------------------------
Main agent input-side tokens                       117,916          19,174
Main agent output tokens                             1,672             584
total_cost_usd (estimate, whole run)                0.0048          0.0113
Models in model_usage (whole run)                        1               1

model_usage keys, main agent alone : ['claude-haiku-5-5']
model_usage keys, with a subagent  : ['claude-haiku-5-5']


## Cell 11: Count where the work happened

The last cell shows how much of the delegated run happened out of the main agent's sight. It walks through the stored messages from the delegated run and sorts them by `parent_tool_use_id`:

- **`None`**: the message belongs to the main agent.
- **Equal to the Agent call's ID**: the message came from inside the subagent.
- **Anything else**: the message came from some other parent, which can happen if a subagent starts another subagent.

For the messages from inside the subagent, the cell also counts which tools the subagent called. Housekeeping messages that do not carry a `parent_tool_use_id` are counted separately by type name, so you can see everything else that appeared in the stream.

By default, the SDK forwards only a subagent's **tool calls and tool results** into your stream. The subagent's own text is not forwarded.

**What to look for:** compare the number of messages from inside the subagent with the number from the main agent. Expect the subagent side to be the larger one, because it did the file reading. That gap is the work the main agent never had to read. It is also the reason delegating agents need careful tracking, because a lot happens in your stream that the main agent never sees.

In [ ]:
from collections import Counter

agent_id = AGENT_CALL.id if AGENT_CALL is not None else None

by_parent = Counter()
subagent_tools = Counter()
other_types = Counter()

for message in deleg_messages:
    if isinstance(message, (AssistantMessage, UserMessage)):
        parent = message.parent_tool_use_id
        if parent is None:
            by_parent["main agent"] += 1
        elif parent == agent_id:
            by_parent["inside the subagent"] += 1
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        subagent_tools[block.name] += 1
        else:
            by_parent["some other parent"] += 1
    else:
        other_types[type(message).__name__] += 1

print("--- Messages in the delegated run, by who produced them ---")
for who in ("main agent", "inside the subagent", "some other parent"):
    print(f"{who:<24}{by_parent[who]:>5}")

print("\n--- Tool calls made inside the subagent ---")
if subagent_tools:
    for tool_name, count in subagent_tools.most_common():
        print(f"{tool_name:<24}{count:>5}")
else:
    print("(none seen)")

print("\n--- Other message types in the stream (no parent_tool_use_id) ---")
for type_name, count in other_types.most_common():
    print(f"{type_name:<28}{count:>5}")

--- Messages in the delegated run, by who produced them ---
main agent                  6
inside the subagent        89
some other parent           0

--- Tool calls made inside the subagent ---
Read                       40
Bash                        2

--- Other message types in the stream (no parent_tool_use_id) ---
TaskProgressMessage            42
SystemMessage                  10
ResultMessage                   2
TaskStartedMessage              1
TaskUpdatedMessage              1
TaskNotificationMessage         1
